# 01 — EDA & Baseline Model

Notebook này thực hiện:
1. Tải và kiểm tra bộ dữ liệu `Steel_industry_data.csv`
2. Phân tích khám phá dữ liệu (EDA): thống kê mô tả, phân phối, tương quan
3. Trực quan hóa chuỗi thời gian tiêu thụ điện
4. Xây dựng mô hình hồi quy/phân loại cơ sở (baseline)

In [ ]:
"""
EDA and baseline modeling.

This script performs exploratory data analysis (EDA) and builds
baseline models for prediction.
"""
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import load_steel_data, inspect_data, clean_data

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Tải dữ liệu

In [ ]:
RAW_PATH = Path('data/raw/Steel_industry_data.csv')
df_raw = load_steel_data(RAW_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head()

## 2. Kiểm tra chất lượng dữ liệu

In [ ]:
report = inspect_data(df_raw)
print('=== Data Inspection Report ===')
for k, v in report.items():
    print(f'{k}: {v}')

## 3. Thống kê mô tả

In [ ]:
df_raw.describe().T.style.background_gradient(cmap='Blues')

## 4. Trực quan hóa chuỗi thời gian và tương quan

In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns.tolist()

fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(14, 3 * len(numeric_cols)))
if len(numeric_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, numeric_cols):
    ax.plot(df_raw['date'], df_raw[col], linewidth=0.7)
    ax.set_title(col)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr = df_raw[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. Làm sạch dữ liệu & Baseline Model

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

df_clean, clean_report = clean_data(df_raw)
print('Clean report:', clean_report)

target = 'Usage_kWh'
feature_cols = [c for c in numeric_cols if c != target]

X = df_clean[feature_cols].fillna(df_clean[feature_cols].median())
y = df_clean[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

models = {
    'LinearRegression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds),
    })

pd.DataFrame(results).set_index('Model').style.highlight_min(subset='MAE', color='lightgreen')